# CI-GCI: Causal-Interventional Grounding & Counterfactual Inpainting for Med-VQA
### End-to-End Autonomous Kaggle Execution Runner
This notebook clones the repository, links datasets, fine-tunes the model on SLAKE and VQA-RAD, evaluates benchmarks, generates proof plots, and prints publication-ready Markdown tables.

### Cell 1: Clone Repository & Install Dependencies

In [ ]:
# Clone or pull latest code from GitHub
!if [ -d "CI-GCI" ]; then cd CI-GCI && git pull origin main; else git clone https://github.com/FaezehMillerAI/CI-GCI.git; fi
%cd CI-GCI

# Install required libraries
!pip install -q torch torchvision transformers scikit-learn matplotlib seaborn pandas tqdm Pillow pyyaml
import torch
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Cell 2: Recursive Dataset Linker
Automatically discovers SLAKE, VQA-RAD, and MS-CXR in Kaggle inputs (`/kaggle/input/`) and links them to local `data/`.

In [ ]:
import os
import glob

os.makedirs("data/slake", exist_ok=True)
os.makedirs("data/VQA-RAD", exist_ok=True)
os.makedirs("data/ms-cxr", exist_ok=True)

# Link SLAKE
slake_jsons = glob.glob("/kaggle/input/**/train.json", recursive=True)
if slake_jsons:
    slake_dir = os.path.dirname(slake_jsons[0])
    print(f"Found SLAKE dataset at: {slake_dir}")
    !ln -sf {slake_dir}/* data/slake/

# Link VQA-RAD
rad_jsons = glob.glob("/kaggle/input/**/*VQA_RAD Dataset Public.json", recursive=True)
if rad_jsons:
    rad_dir = os.path.dirname(rad_jsons[0])
    print(f"Found VQA-RAD dataset at: {rad_dir}")
    !ln -sf {rad_dir}/* data/VQA-RAD/

print("Linked Datasets in data/:", os.listdir("data"))

### Cell 3: Verify Pipeline Integrity

In [ ]:
!PYTHONPATH=. python3 scripts/verify_pipeline.py

### Cell 4: Fine-Tune CI-GCI Model on SLAKE (15 Epochs)

In [ ]:
!PYTHONPATH=. python3 training/train_slake_vqa.py --dataset slake --epochs 15 --batch_size 16 --lr 5e-4 --device cuda

### Cell 5: Fine-Tune CI-GCI Model on VQA-RAD (15 Epochs)

In [ ]:
!PYTHONPATH=. python3 training/train_slake_vqa.py --dataset vqa_rad --epochs 15 --batch_size 16 --lr 5e-4 --device cuda

### Cell 5b: Fine-Tune CI-GCI Model on PathVQA & Update Benchmarks (10 Epochs)
Trains  on histopathology closed QA pairs, generates test records, and updates canonical tables.

In [ ]:
!PYTHONPATH=. python3 scripts/train_and_eval_pathvqa.py --epochs 10 --batch_size 16 --lr 5e-4 --device cuda

### Cell 6: Run Comparative Benchmarks (SLAKE & VQA-RAD)

In [ ]:
print("=== BENCHMARKING SLAKE ===")
!PYTHONPATH=. python3 scripts/benchmark_comparison.py --dataset slake --device cuda

print("\n=== BENCHMARKING VQA-RAD ===")
!PYTHONPATH=. python3 scripts/benchmark_comparison.py --dataset vqa_rad --device cuda

### Cell 7: Generate Proof Plots & Reliability Diagrams

In [ ]:
!PYTHONPATH=. python3 scripts/generate_plots_and_proofs.py --device cuda

### Cell 8: Generate & Display 7 Publication-Ready Markdown Tables

In [ ]:
!PYTHONPATH=. python3 evaluation/result_table_generator.py

import os
tables_dir = "outputs/tables"
if os.path.exists(tables_dir):
    for fname in sorted(os.listdir(tables_dir)):
        if fname.endswith(".md"):
            fpath = os.path.join(tables_dir, fname)
            print("="*80)
            print(f" DISPLAYING TABLE: {fname.upper()}")
            print("="*80)
            with open(fpath, "r") as f:
                print(f.read())
            print("\n")